# Prologue -- Preprocessing PHANGS

In this Notebook, we show how the example data file, `NGC1385_metals.fits`, is generated from real galaxy data. Specifically, this data file was produced from the emission line maps of NGC 1385 taken from the PHANGS-MUSE survey, first released in 2021.

## Outline

We will process our data in 5 steps:

1. Download the data
2. Perform a signal to noise cut to mask out noisy spaxels (spatial pixels)
3. Perform an extinction correction to negate the effects of galactic dust
4. Remove spaxels that are contaminated by diffuse ionised gas (DIG)
5. Calculate the metallicity of each spaxel.

But before we proceed, we will import our necessary packages:

In [19]:
import numpy as np
import geogals as gg
from astropy.io import fits

### Step 1: Downloading our data

First, we must download the raw dataset. This can be done from [the PHANGS website](https://sites.google.com/view/phangs/data?authuser=0). The MUSE data we are interested in can be found [here](https://www.canfar.net/storage/vault/list/phangs/RELEASES/PHANGS-MUSE/DR1.0/MAPS). I this example, the only file that we need is `NGC1385_MAPS_native.fits`.

Once that is downloaded, use `astropy` to open the data.

In [20]:
data_path = '../../Data/' # location at which the file is saved
init_hdu_list = fits.open(data_path + 'NGC1385_MAPS_native.fits')

# To homogenise our data, we remove the first HDU
gal_df = init_hdu_list[1:]

### 2. Perform a S/N cut

Fortunately, there's a function for that. This directly modifies the data it is used on and returns nothing. Here we use a S/N threshold of 10:

In [16]:
gg.phangs.SN_cut(gal_df, 10)

NameError: name 'np' is not defined

### 3. Perform an extinction correction.

Luckily, most of this work has already been done for us, as Cardelli, Clayton and Mathis ot only wrote [an excellent and highly cited paper](https://ui.adsabs.harvard.edu/abs/1989ApJ...345..245C/abstract) about how to do this, but also published a Python package, `extinction`, implementing their equations. In GeoGals, the function `extinction_correction` acts as a wrapper around this code, applying it to every HDU in our list. However, to use this, we must also supply this function with a list of the wavelengths of all lines observed in the MUSE survey:

In [18]:
wavelengths = np.array([4861.3, 4958, 5006.0, 6548.0, 6562.0, 6583.0, 6716.0, 6731.0]) # angstroms

gg.phangs.extinction_correction(gal_df, wavelengths)

NameError: name 'np' is not defined

### 4. Mask out DIG regions

Diffuse ionised gas, or DIG, is a real nuisance.

Two seminal works

Each of these functions returns an array of the same size as our input data

In [ ]:
S2_BPT_classification = gg.phangs.classify_S2_BPT(gal_df)
N2_BPT_classification = gg.phangs.classify_N2_BPT(gal_df)
combo_classification  = S2_BPT_classification*N2_BPT_classification

# Turn regions which are considered DIG dominated into nans
is_DIG = ~combo_classification

gal_df[is_DIG] = np.nan